This notebook is intended for class balancing (main disorder and specific disorder) using CTGAN, based on the dataset by Park et al. (2021).

## | Importing libraries

In [1]:
!pip install -q ctgan;
# !pip install -q table_evaluator;
!pip install sdmetrics;

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 61.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcugraph-cu12 24.12.0 requires pylibraft-cu12==24.12.*, but you have pylibraft-cu12 25.2.0 which is incompa

In [2]:
from ctgan import CTGAN
import sdmetrics
from sklearn.impute import KNNImputer
import torch
import pandas as pd
import logging
import time

from warnings import filterwarnings
filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.init()
torch.zeros(1).cuda()

torch.cuda.is_available(), device, torch.cuda.get_device_name(0)

(True, device(type='cuda'), 'Tesla T4')

## | Dataset

In [3]:
df = pd.read_csv('../input/eeg-psychiatric-disorders-dataset/EEG.machinelearing_data_BRMH.csv')
df

,no.,sex,age,eeg.date,education,IQ,main.disorder,specific.disorder,AB.A.delta.a.FP1,AB.A.delta.b.FP2,...,COH.F.gamma.o.Pz.p.P4,COH.F.gamma.o.Pz.q.T6,COH.F.gamma.o.Pz.r.O1,COH.F.gamma.o.Pz.s.O2,COH.F.gamma.p.P4.q.T6,COH.F.gamma.p.P4.r.O1,COH.F.gamma.p.P4.s.O2,COH.F.gamma.q.T6.r.O1,COH.F.gamma.q.T6.s.O2,COH.F.gamma.r.O1.s.O2
0,1,M,57.0,2012.8.30,NaN,NaN,Addictive disorder,Alcohol use disorder,35.998557,21.717375,...,55.989192,16.739679,23.452271,45.678820,30.167520,16.918761,48.850427,9.422630,34.507082,28.613029
1,2,M,37.0,2012.9.6,6.0,120.0,Addictive disorder,Alcohol use disorder,13.425118,11.002916,...,45.595619,17.510824,26.777368,28.201062,57.108861,32.375401,60.351749,13.900981,57.831848,43.463261
2,3,M,32.0,2012.9.10,16.0,113.0,Addictive disorder,Alcohol use disorder,29.941780,27.544684,...,99.475453,70.654171,39.131547,69.920996,71.063644,38.534505,69.908764,27.180532,64.803155,31.485799
3,4,M,35.0,2012.10.8,18.0,126.0,Addictive disorder,Alcohol use disorder,21.496226,21.846832,...,59.986561,63.822201,36.478254,47.117006,84.658376,24.724096,50.299349,35.319695,79.822944,41.141873
4,5,M,36.0,2012.10.18,16.0,112.0,Addictive disorder,Alcohol use disorder,37.775667,33.607679,...,61.462720,59.166097,51.465531,58.635415,80.685608,62.138436,75.888749,61.003944,87.455509,70.531662
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
940,941,M,22.0,2014.8.28,13.0,116.0,Healthy control,Healthy control,41.851823,36.771496,...,82.905657,34.850706,63.970519,63.982003,51.244725,62.203684,62.062237,31.013031,31.183413,98.325230
941,942,M,26.0,2014.9.19,13.0,118.0,Healthy control,Healthy control,18.986856,19.401387,...,65.917918,66.700117,44.756285,49.787513,98.905995,54.021304,93.902401,52.740396,92.807331,56.320868
942,943,M,26.0,2014.9.27,16.0,113.0,Healthy control,Healthy control,28.781317,32.369230,...,61.040959,27.632209,45.552852,33.638817,46.690983,19.382928,41.050717,7.045821,41.962451,19.092111
943,944,M,24.0,2014.9.20,13.0,107.0,Healthy control,Healthy control,19.929100,25.196375,...,99.113664,48.328934,41.248470,28.192238,48.665743,42.007147,28.735945,27.176500,27.529522,20.028446


## | Data Cleaning

In [4]:
def remove_missing_columns(df, threshold=0.5):
        limit = int(threshold * len(df))
        df = df.dropna(thresh=limit, axis=1)
        return df

def analyze_missing_values(df):
    missing_values = df.isnull().sum()
    missing_values = missing_values[missing_values > 0]
    total_number_nans = df.isnull().sum().sum()
    
    return missing_values, total_number_nans

def handle_nans(df):
    columns_with_nans = df.columns[df.isnull().any()].tolist()
    
    knn_imputer = KNNImputer(n_neighbors=5, weights='uniform', metric='nan_euclidean')
    
    df_imputed = pd.DataFrame(knn_imputer.fit_transform(df[columns_with_nans]),
                              columns=columns_with_nans)
    
    df[columns_with_nans] = df_imputed[columns_with_nans]

    return df

In [5]:
analyze_missing_values(df)

(education        15
 IQ               13
 Unnamed: 122    945
 dtype: int64,
 973)

In [6]:
df = remove_missing_columns(df)
analyze_missing_values(df)

(education    15
 IQ           13
 dtype: int64,
 28)

In [7]:
df = handle_nans(df)
analyze_missing_values(df)

(Series([], dtype: int64), 0)

In [8]:
df

,no.,sex,age,eeg.date,education,IQ,main.disorder,specific.disorder,AB.A.delta.a.FP1,AB.A.delta.b.FP2,...,COH.F.gamma.o.Pz.p.P4,COH.F.gamma.o.Pz.q.T6,COH.F.gamma.o.Pz.r.O1,COH.F.gamma.o.Pz.s.O2,COH.F.gamma.p.P4.q.T6,COH.F.gamma.p.P4.r.O1,COH.F.gamma.p.P4.s.O2,COH.F.gamma.q.T6.r.O1,COH.F.gamma.q.T6.s.O2,COH.F.gamma.r.O1.s.O2
0,1,M,57.0,2012.8.30,13.43871,101.580472,Addictive disorder,Alcohol use disorder,35.998557,21.717375,...,55.989192,16.739679,23.452271,45.678820,30.167520,16.918761,48.850427,9.422630,34.507082,28.613029
1,2,M,37.0,2012.9.6,6.00000,120.000000,Addictive disorder,Alcohol use disorder,13.425118,11.002916,...,45.595619,17.510824,26.777368,28.201062,57.108861,32.375401,60.351749,13.900981,57.831848,43.463261
2,3,M,32.0,2012.9.10,16.00000,113.000000,Addictive disorder,Alcohol use disorder,29.941780,27.544684,...,99.475453,70.654171,39.131547,69.920996,71.063644,38.534505,69.908764,27.180532,64.803155,31.485799
3,4,M,35.0,2012.10.8,18.00000,126.000000,Addictive disorder,Alcohol use disorder,21.496226,21.846832,...,59.986561,63.822201,36.478254,47.117006,84.658376,24.724096,50.299349,35.319695,79.822944,41.141873
4,5,M,36.0,2012.10.18,16.00000,112.000000,Addictive disorder,Alcohol use disorder,37.775667,33.607679,...,61.462720,59.166097,51.465531,58.635415,80.685608,62.138436,75.888749,61.003944,87.455509,70.531662
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
940,941,M,22.0,2014.8.28,13.00000,116.000000,Healthy control,Healthy control,41.851823,36.771496,...,82.905657,34.850706,63.970519,63.982003,51.244725,62.203684,62.062237,31.013031,31.183413,98.325230
941,942,M,26.0,2014.9.19,13.00000,118.000000,Healthy control,Healthy control,18.986856,19.401387,...,65.917918,66.700117,44.756285,49.787513,98.905995,54.021304,93.902401,52.740396,92.807331,56.320868
942,943,M,26.0,2014.9.27,16.00000,113.000000,Healthy control,Healthy control,28.781317,32.369230,...,61.040959,27.632209,45.552852,33.638817,46.690983,19.382928,41.050717,7.045821,41.962451,19.092111
943,944,M,24.0,2014.9.20,13.00000,107.000000,Healthy control,Healthy control,19.929100,25.196375,...,99.113664,48.328934,41.248470,28.192238,48.665743,42.007147,28.735945,27.176500,27.529522,20.028446


In [9]:
# (19 * 6) + (171 * 6) = 1140

df.iloc[:, 8:]

,AB.A.delta.a.FP1,AB.A.delta.b.FP2,AB.A.delta.c.F7,AB.A.delta.d.F3,AB.A.delta.e.Fz,AB.A.delta.f.F4,AB.A.delta.g.F8,AB.A.delta.h.T3,AB.A.delta.i.C3,AB.A.delta.j.Cz,...,COH.F.gamma.o.Pz.p.P4,COH.F.gamma.o.Pz.q.T6,COH.F.gamma.o.Pz.r.O1,COH.F.gamma.o.Pz.s.O2,COH.F.gamma.p.P4.q.T6,COH.F.gamma.p.P4.r.O1,COH.F.gamma.p.P4.s.O2,COH.F.gamma.q.T6.r.O1,COH.F.gamma.q.T6.s.O2,COH.F.gamma.r.O1.s.O2
0,35.998557,21.717375,21.518280,26.825048,26.611516,25.732649,16.563408,29.891368,22.402246,22.582176,...,55.989192,16.739679,23.452271,45.678820,30.167520,16.918761,48.850427,9.422630,34.507082,28.613029
1,13.425118,11.002916,11.942516,15.272216,14.151570,12.456034,8.436832,9.975238,14.834740,10.950564,...,45.595619,17.510824,26.777368,28.201062,57.108861,32.375401,60.351749,13.900981,57.831848,43.463261
2,29.941780,27.544684,17.150159,23.608960,27.087811,13.541237,16.523963,12.775574,21.686306,18.367666,...,99.475453,70.654171,39.131547,69.920996,71.063644,38.534505,69.908764,27.180532,64.803155,31.485799
3,21.496226,21.846832,17.364316,13.833701,14.100954,13.100939,14.613650,8.063191,11.015078,11.639560,...,59.986561,63.822201,36.478254,47.117006,84.658376,24.724096,50.299349,35.319695,79.822944,41.141873
4,37.775667,33.607679,21.865556,21.771413,22.854536,21.456377,15.969042,9.434306,15.244523,17.041979,...,61.462720,59.166097,51.465531,58.635415,80.685608,62.138436,75.888749,61.003944,87.455509,70.531662
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
940,41.851823,36.771496,43.671792,36.860889,24.732236,23.607823,23.288260,7.520642,17.636528,20.220791,...,82.905657,34.850706,63.970519,63.982003,51.244725,62.203684,62.062237,31.013031,31.183413,98.325230
941,18.986856,19.401387,27.586436,20.194732,19.407491,20.216570,16.465027,13.178851,12.687296,20.257619,...,65.917918,66.700117,44.756285,49.787513,98.905995,54.021304,93.902401,52.740396,92.807331,56.320868
942,28.781317,32.369230,11.717778,23.134370,26.209302,25.484497,22.586688,11.368466,21.799254,36.083181,...,61.040959,27.632209,45.552852,33.638817,46.690983,19.382928,41.050717,7.045821,41.962451,19.092111
943,19.929100,25.196375,14.445391,16.453456,16.590649,16.007279,18.909188,13.438102,17.442777,18.859586,...,99.113664,48.328934,41.248470,28.192238,48.665743,42.007147,28.735945,27.176500,27.529522,20.028446


In [10]:
target = 'specific.disorder'
# label_main = 'Addictive disorder'
label_specific = 'Alcohol use disorder'

df[target]

0      Alcohol use disorder
1      Alcohol use disorder
2      Alcohol use disorder
3      Alcohol use disorder
4      Alcohol use disorder
               ...         
940         Healthy control
941         Healthy control
942         Healthy control
943         Healthy control
944         Healthy control
Name: specific.disorder, Length: 945, dtype: object

In [11]:
y = df[target]
X = df.iloc[:,8:]
quantitative_features = df[['age','education', 'IQ']]

X = pd.concat([quantitative_features, X], axis=1)
X

,age,education,IQ,AB.A.delta.a.FP1,AB.A.delta.b.FP2,AB.A.delta.c.F7,AB.A.delta.d.F3,AB.A.delta.e.Fz,AB.A.delta.f.F4,AB.A.delta.g.F8,...,COH.F.gamma.o.Pz.p.P4,COH.F.gamma.o.Pz.q.T6,COH.F.gamma.o.Pz.r.O1,COH.F.gamma.o.Pz.s.O2,COH.F.gamma.p.P4.q.T6,COH.F.gamma.p.P4.r.O1,COH.F.gamma.p.P4.s.O2,COH.F.gamma.q.T6.r.O1,COH.F.gamma.q.T6.s.O2,COH.F.gamma.r.O1.s.O2
0,57.0,13.43871,101.580472,35.998557,21.717375,21.518280,26.825048,26.611516,25.732649,16.563408,...,55.989192,16.739679,23.452271,45.678820,30.167520,16.918761,48.850427,9.422630,34.507082,28.613029
1,37.0,6.00000,120.000000,13.425118,11.002916,11.942516,15.272216,14.151570,12.456034,8.436832,...,45.595619,17.510824,26.777368,28.201062,57.108861,32.375401,60.351749,13.900981,57.831848,43.463261
2,32.0,16.00000,113.000000,29.941780,27.544684,17.150159,23.608960,27.087811,13.541237,16.523963,...,99.475453,70.654171,39.131547,69.920996,71.063644,38.534505,69.908764,27.180532,64.803155,31.485799
3,35.0,18.00000,126.000000,21.496226,21.846832,17.364316,13.833701,14.100954,13.100939,14.613650,...,59.986561,63.822201,36.478254,47.117006,84.658376,24.724096,50.299349,35.319695,79.822944,41.141873
4,36.0,16.00000,112.000000,37.775667,33.607679,21.865556,21.771413,22.854536,21.456377,15.969042,...,61.462720,59.166097,51.465531,58.635415,80.685608,62.138436,75.888749,61.003944,87.455509,70.531662
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
940,22.0,13.00000,116.000000,41.851823,36.771496,43.671792,36.860889,24.732236,23.607823,23.288260,...,82.905657,34.850706,63.970519,63.982003,51.244725,62.203684,62.062237,31.013031,31.183413,98.325230
941,26.0,13.00000,118.000000,18.986856,19.401387,27.586436,20.194732,19.407491,20.216570,16.465027,...,65.917918,66.700117,44.756285,49.787513,98.905995,54.021304,93.902401,52.740396,92.807331,56.320868
942,26.0,16.00000,113.000000,28.781317,32.369230,11.717778,23.134370,26.209302,25.484497,22.586688,...,61.040959,27.632209,45.552852,33.638817,46.690983,19.382928,41.050717,7.045821,41.962451,19.092111
943,24.0,13.00000,107.000000,19.929100,25.196375,14.445391,16.453456,16.590649,16.007279,18.909188,...,99.113664,48.328934,41.248470,28.192238,48.665743,42.007147,28.735945,27.176500,27.529522,20.028446


In [12]:
df = pd.concat([X, y], axis=1)
df

,age,education,IQ,AB.A.delta.a.FP1,AB.A.delta.b.FP2,AB.A.delta.c.F7,AB.A.delta.d.F3,AB.A.delta.e.Fz,AB.A.delta.f.F4,AB.A.delta.g.F8,...,COH.F.gamma.o.Pz.q.T6,COH.F.gamma.o.Pz.r.O1,COH.F.gamma.o.Pz.s.O2,COH.F.gamma.p.P4.q.T6,COH.F.gamma.p.P4.r.O1,COH.F.gamma.p.P4.s.O2,COH.F.gamma.q.T6.r.O1,COH.F.gamma.q.T6.s.O2,COH.F.gamma.r.O1.s.O2,specific.disorder
0,57.0,13.43871,101.580472,35.998557,21.717375,21.518280,26.825048,26.611516,25.732649,16.563408,...,16.739679,23.452271,45.678820,30.167520,16.918761,48.850427,9.422630,34.507082,28.613029,Alcohol use disorder
1,37.0,6.00000,120.000000,13.425118,11.002916,11.942516,15.272216,14.151570,12.456034,8.436832,...,17.510824,26.777368,28.201062,57.108861,32.375401,60.351749,13.900981,57.831848,43.463261,Alcohol use disorder
2,32.0,16.00000,113.000000,29.941780,27.544684,17.150159,23.608960,27.087811,13.541237,16.523963,...,70.654171,39.131547,69.920996,71.063644,38.534505,69.908764,27.180532,64.803155,31.485799,Alcohol use disorder
3,35.0,18.00000,126.000000,21.496226,21.846832,17.364316,13.833701,14.100954,13.100939,14.613650,...,63.822201,36.478254,47.117006,84.658376,24.724096,50.299349,35.319695,79.822944,41.141873,Alcohol use disorder
4,36.0,16.00000,112.000000,37.775667,33.607679,21.865556,21.771413,22.854536,21.456377,15.969042,...,59.166097,51.465531,58.635415,80.685608,62.138436,75.888749,61.003944,87.455509,70.531662,Alcohol use disorder
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
940,22.0,13.00000,116.000000,41.851823,36.771496,43.671792,36.860889,24.732236,23.607823,23.288260,...,34.850706,63.970519,63.982003,51.244725,62.203684,62.062237,31.013031,31.183413,98.325230,Healthy control
941,26.0,13.00000,118.000000,18.986856,19.401387,27.586436,20.194732,19.407491,20.216570,16.465027,...,66.700117,44.756285,49.787513,98.905995,54.021304,93.902401,52.740396,92.807331,56.320868,Healthy control
942,26.0,16.00000,113.000000,28.781317,32.369230,11.717778,23.134370,26.209302,25.484497,22.586688,...,27.632209,45.552852,33.638817,46.690983,19.382928,41.050717,7.045821,41.962451,19.092111,Healthy control
943,24.0,13.00000,107.000000,19.929100,25.196375,14.445391,16.453456,16.590649,16.007279,18.909188,...,48.328934,41.248470,28.192238,48.665743,42.007147,28.735945,27.176500,27.529522,20.028446,Healthy control


In [13]:
df[target]

0      Alcohol use disorder
1      Alcohol use disorder
2      Alcohol use disorder
3      Alcohol use disorder
4      Alcohol use disorder
               ...         
940         Healthy control
941         Healthy control
942         Healthy control
943         Healthy control
944         Healthy control
Name: specific.disorder, Length: 945, dtype: object

In [14]:
class_counts = df[target].value_counts()
max_class_size = class_counts.max()

class_counts, max_class_size

(specific.disorder
 Depressive disorder               199
 Schizophrenia                     117
 Healthy control                    95
 Alcohol use disorder               93
 Behavioral addiction disorder      93
 Bipolar disorder                   67
 Panic disorder                     59
 Posttraumatic stress disorder      52
 Social anxiety disorder            48
 Obsessive compulsitve disorder     46
 Acute stress disorder              38
 Adjustment disorder                38
 Name: count, dtype: int64,
 199)

In [15]:
synthetic_samples = []

In [16]:
df[df[target] == label_specific]

,age,education,IQ,AB.A.delta.a.FP1,AB.A.delta.b.FP2,AB.A.delta.c.F7,AB.A.delta.d.F3,AB.A.delta.e.Fz,AB.A.delta.f.F4,AB.A.delta.g.F8,...,COH.F.gamma.o.Pz.q.T6,COH.F.gamma.o.Pz.r.O1,COH.F.gamma.o.Pz.s.O2,COH.F.gamma.p.P4.q.T6,COH.F.gamma.p.P4.r.O1,COH.F.gamma.p.P4.s.O2,COH.F.gamma.q.T6.r.O1,COH.F.gamma.q.T6.s.O2,COH.F.gamma.r.O1.s.O2,specific.disorder
0,57.00,13.43871,101.580472,35.998557,21.717375,21.518280,26.825048,26.611516,25.732649,16.563408,...,16.739679,23.452271,45.678820,30.167520,16.918761,48.850427,9.422630,34.507082,28.613029,Alcohol use disorder
1,37.00,6.00000,120.000000,13.425118,11.002916,11.942516,15.272216,14.151570,12.456034,8.436832,...,17.510824,26.777368,28.201062,57.108861,32.375401,60.351749,13.900981,57.831848,43.463261,Alcohol use disorder
2,32.00,16.00000,113.000000,29.941780,27.544684,17.150159,23.608960,27.087811,13.541237,16.523963,...,70.654171,39.131547,69.920996,71.063644,38.534505,69.908764,27.180532,64.803155,31.485799,Alcohol use disorder
3,35.00,18.00000,126.000000,21.496226,21.846832,17.364316,13.833701,14.100954,13.100939,14.613650,...,63.822201,36.478254,47.117006,84.658376,24.724096,50.299349,35.319695,79.822944,41.141873,Alcohol use disorder
4,36.00,16.00000,112.000000,37.775667,33.607679,21.865556,21.771413,22.854536,21.456377,15.969042,...,59.166097,51.465531,58.635415,80.685608,62.138436,75.888749,61.003944,87.455509,70.531662,Alcohol use disorder
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
861,37.54,16.00000,111.000000,12.887892,12.811554,11.271938,12.730815,13.449244,13.440774,9.525110,...,59.890110,60.006472,76.000963,76.940229,51.995874,83.445173,29.867826,64.298916,61.308432,Alcohol use disorder
867,41.03,12.00000,94.000000,15.672306,15.865889,14.027073,16.038039,15.950864,15.901984,15.272183,...,42.022839,44.299151,58.055661,65.048325,33.605385,65.283460,24.899975,70.553439,54.170720,Alcohol use disorder
876,54.52,6.00000,68.000000,17.581980,17.776402,13.869724,14.151970,20.869926,18.369719,18.315985,...,48.282117,73.185133,66.742485,49.466312,59.270167,66.639938,40.297206,55.547526,63.630547,Alcohol use disorder
894,49.01,10.00000,106.000000,16.224258,14.015313,10.946295,15.361129,18.104099,13.576599,11.332519,...,30.140302,38.413929,66.371912,60.563340,21.307001,84.886865,9.804814,63.434182,33.162128,Alcohol use disorder


In [17]:
start = time.time()

for label, count in class_counts.items():
    if count < max_class_size:
        print(f"Training for class: {label}")
        
        minority_data = df[df[target] == label]

        ctgan = CTGAN(epochs=300)

        ctgan.fit(minority_data, [target])

        n_to_generate = max_class_size - count
        new_data = ctgan.sample(n_to_generate)

        synthetic_samples.append(new_data)
        
        print(f"Class {label} completed. {n_to_generate} samples generated.")

end = time.time()
elapsed = end - start
print(f"Total training time: {int(elapsed // 60)} min {int(elapsed % 60)} s")

Training for class: Schizophrenia
Class Schizophrenia completed. 82 samples generated.
Training for class: Healthy control
Class Healthy control completed. 104 samples generated.
Training for class: Alcohol use disorder
Class Alcohol use disorder completed. 106 samples generated.
Training for class: Behavioral addiction disorder
Class Behavioral addiction disorder completed. 106 samples generated.
Training for class: Bipolar disorder
Class Bipolar disorder completed. 132 samples generated.
Training for class: Panic disorder
Class Panic disorder completed. 140 samples generated.
Training for class: Posttraumatic stress disorder
Class Posttraumatic stress disorder completed. 147 samples generated.
Training for class: Social anxiety disorder
Class Social anxiety disorder completed. 151 samples generated.
Training for class: Obsessive compulsitve disorder
Class Obsessive compulsitve disorder completed. 153 samples generated.
Training for class: Acute stress disorder
Class Acute stress diso

In [18]:
synthetic_samples

[          age  education          IQ  AB.A.delta.a.FP1  AB.A.delta.b.FP2  \
 0   12.486723  12.720248  111.652880          2.725357         21.452343   
 1   18.546245  14.794858   91.973729          4.684256         35.090543   
 2   13.599379  11.907947  107.347258          9.349618         19.421081   
 3   27.346908  12.815714   82.375051         34.597320         50.193828   
 4   15.558836  19.125513  108.652151         -1.817344         29.857772   
 ..        ...        ...         ...               ...               ...   
 77  54.045217  18.654616   77.411420          5.550627         17.067873   
 78  29.643875  19.991001   79.396465         20.670457         16.384945   
 79  21.986144  12.342485   95.601108         12.705330         19.793787   
 80  25.295576  13.538218   90.665064          7.208372         27.622288   
 81  14.928677  12.477826   87.374928          7.066886         45.419169   
 
     AB.A.delta.c.F7  AB.A.delta.d.F3  AB.A.delta.e.Fz  AB.A.delta.f.F4  \

In [19]:
synthetic_samples = pd.concat(synthetic_samples, ignore_index=True)

In [20]:
synthetic_samples

,age,education,IQ,AB.A.delta.a.FP1,AB.A.delta.b.FP2,AB.A.delta.c.F7,AB.A.delta.d.F3,AB.A.delta.e.Fz,AB.A.delta.f.F4,AB.A.delta.g.F8,...,COH.F.gamma.o.Pz.q.T6,COH.F.gamma.o.Pz.r.O1,COH.F.gamma.o.Pz.s.O2,COH.F.gamma.p.P4.q.T6,COH.F.gamma.p.P4.r.O1,COH.F.gamma.p.P4.s.O2,COH.F.gamma.q.T6.r.O1,COH.F.gamma.q.T6.s.O2,COH.F.gamma.r.O1.s.O2,specific.disorder
0,12.486723,12.720248,111.652880,2.725357,21.452343,27.299812,19.019614,25.373663,22.539893,14.994883,...,26.880195,57.820169,72.340303,113.228991,56.707236,54.023712,5.539792,38.782711,-5.282597,Schizophrenia
1,18.546245,14.794858,91.973729,4.684256,35.090543,25.702844,21.747822,20.096341,6.902994,9.496179,...,45.567550,25.359201,81.145683,97.401796,1.298894,83.881808,24.405675,13.098263,40.959226,Schizophrenia
2,13.599379,11.907947,107.347258,9.349618,19.421081,31.597079,24.319588,11.128257,1.524109,10.764051,...,11.019094,31.597541,48.291539,111.414746,56.116052,39.928965,19.604073,24.049729,79.182487,Schizophrenia
3,27.346908,12.815714,82.375051,34.597320,50.193828,8.127347,23.540958,30.025157,17.017312,0.138230,...,33.195094,1.373304,56.348701,98.378207,-3.453199,69.421351,-9.791777,9.909273,83.766819,Schizophrenia
4,15.558836,19.125513,108.652151,-1.817344,29.857772,20.471747,24.993812,15.728950,18.494554,2.248556,...,40.036583,31.316394,65.225965,62.629056,14.402702,5.156964,1.741342,41.310525,53.742409,Schizophrenia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1438,41.024158,21.338190,137.508387,3.897484,6.600202,11.625845,13.960831,37.676382,30.945274,11.202927,...,47.175044,25.935223,80.153058,110.550349,58.663091,41.690877,-5.156744,81.195055,2.983596,Adjustment disorder
1439,55.499292,12.990695,80.726012,1.785503,8.328488,30.755002,21.065130,37.601797,26.809310,29.847553,...,58.018386,62.111926,86.762645,62.187134,16.427574,66.377102,18.068607,46.783253,44.923293,Adjustment disorder
1440,56.778344,18.878593,138.331332,19.287896,0.594376,41.444599,41.514412,41.861253,21.975527,26.688986,...,87.730085,66.344053,85.630108,85.603845,1.485398,49.566798,24.463577,111.231455,93.531027,Adjustment disorder
1441,46.136011,15.122581,121.912296,2.706127,25.875806,38.546985,24.760639,8.782951,20.560540,15.400793,...,72.473839,80.167055,86.254430,59.050772,45.043488,80.776484,-13.517152,99.023455,3.971395,Adjustment disorder


In [21]:
class_counts_synth = synthetic_samples[target].value_counts()
max_class_size_synth = class_counts_synth.max()

# [82, 104, 106, 106, 132, 140, 147, 151, 153, 161, 161] = 1.443
class_counts_synth, max_class_size_synth, class_counts_synth.sum()

(specific.disorder
 Acute stress disorder             161
 Adjustment disorder               161
 Obsessive compulsitve disorder    153
 Social anxiety disorder           151
 Posttraumatic stress disorder     147
 Panic disorder                    140
 Bipolar disorder                  132
 Alcohol use disorder              106
 Behavioral addiction disorder     106
 Healthy control                   104
 Schizophrenia                      82
 Name: count, dtype: int64,
 161,
 1443)

In [22]:
synthetic_samples.to_csv('synthetic_samples_specific_disorder_ctgan.csv', index=True)